# Diffusion-wave Caputo derivative validation suite

This notebook reproduces and extends the MCfd-style validation for the four schemes `MC-I`, `MC-II`, `GJ-I`, and `GJ-II` for $lpha\in(1,2)$.

It focuses on three diagnostics:

1. the influence of $lpha$, especially $lpha	o2$;
2. the influence of quadrature size $M$, including why Gauss--Jacobi errors can increase for large $M$ in raw quotient implementations;
3. non-smooth test functions, where the $C^2$ / analytic assumptions used by the theory are violated.

The implementation is in `dw_fractional_validation_suite.py`; this notebook is a thin, editable driver.

In [ ]:
from pathlib import Path
import importlib
import dw_fractional_validation_suite as dvs

importlib.reload(dvs)
dvs.set_nature_style()

outdir = Path('dw_validation_outputs')
outdir.mkdir(exist_ok=True)
seed = 229

## 1. $\alpha$ sweep

Smooth benchmark: $f(t)=e^{-t}$, with exact Caputo derivative

$$
D_t^\alpha e^{\lambda t}=\lambda^2 t^{2-\alpha}E_{1,3-\alpha}(\lambda t).
$$

The third panel tracks the endpoint mass $P(\tau<\epsilon/t)$ for $\tau\sim\mathrm{Beta}(2-\alpha,1)$.  This is the main Monte Carlo failure mode as $\alpha\to2$.

In [ ]:
alpha_results = dvs.experiment_alpha_sweep(
    outdir=outdir,
    seed=seed,
    t=1.5,
    lam=-1.0,
    M_mc=10_000,
    M_gj=100,
    eps_abs=1e-7,
)
print(outdir / 'fig_alpha_sweep.pdf')

## 2. $M$ sweep

This experiment separates two effects:

- Monte Carlo sampling error, summarized by median and interquartile bands over repeated runs.
- Gauss--Jacobi raw quotient conditioning.  The `stable` curves use algebraically equivalent `expm1`/Taylor evaluations for the exponential benchmark, showing that the large-$M$ increase is a floating-point/raw-quotient issue rather than a contradiction of the quadrature theory.

In [ ]:
M_results = dvs.experiment_M_sweep(
    outdir=outdir,
    seed=seed,
    t=0.75,
    alpha=1.5,
    lam=-1.0,
    eps_abs=1e-7,
    mc_repeats=32,
)
print(outdir / 'fig_M_sweep.pdf')

## 3. Non-smooth functions

The non-smooth benchmark is

$$
f(t)=(t-t_c)_+^\beta,\qquad 1<\beta<2,
$$

which is $C^1$ but not $C^2$ at $t=t_c$.  Its Caputo derivative is still available in closed form for $t>t_c$:

$$
D_t^\alpha (t-t_c)_+^\beta=\frac{\Gamma(\beta+1)}{\Gamma(\beta+1-\alpha)}(t-t_c)^{\beta-\alpha}.
$$

This test directly probes what happens when the smoothness assumptions behind the Gauss--Jacobi spectral/algebraic estimates are weakened.

In [ ]:
nonsmooth_results = dvs.experiment_nonsmooth(
    outdir=outdir,
    seed=seed,
    t=0.75,
    alpha=1.5,
    beta=1.35,
    tc=0.25,
)
print(outdir / 'fig_nonsmooth.pdf')

## Files

The script writes PDF/PNG figures and a `validation_results.json` file that stores the numerical data used in the figures.

In [ ]:
for p in sorted(outdir.glob('*')):
    print(p)